# C4 · Bash intermedio y manejo de errores

**Curso:** Bioinformática y Biología Computacional · Universidad EAFIT  
**Duración sugerida:** 3 horas  
**Modalidad:** explicación breve → práctica guiada → reto → evidencia reproducible

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UniversidadEAFIT/compubiol_course/blob/master/notebooks/04_bash_robusto/04_bash_manejo_errores.ipynb)

> **Continuidad del material histórico:** esta versión reemplaza y amplía `20231/LinuxII/LinuxII_Ejercicios.ipynb + 20231/Error_handling.pdf`.

## Pregunta guía

Una secuencia de comandos manuales funciona para una muestra, pero debe procesar una carpeta completa y detenerse si una entrada es inválida. **¿Qué convierte un conjunto de comandos en una herramienta confiable?**

### Objetivos

- usar variables, parámetros, bucles, condicionales y funciones;
- comprender códigos de salida y cortocircuitos `&&`/`||`;
- aplicar `set -Eeuo pipefail` y `trap`;
- validar entradas antes de producir resultados;
- registrar progreso y fallos en un log;
- resumir FASTA/FASTQ con AWK, `sort` y `uniq`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import importlib.util

REPO_URL = "https://github.com/UniversidadEAFIT/compubiol_course.git"
COLAB_DIR = Path("/content/compubiol_course")

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB and importlib.util.find_spec("Bio") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)

if IN_COLAB and not COLAB_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(COLAB_DIR)], check=True)
    os.chdir(COLAB_DIR)

start = Path.cwd().resolve()
ROOT = next((p for p in [start, *start.parents] if (p / "data").is_dir() and (p / "notebooks").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError(
        "No se encontró la raíz del curso. Ejecute el notebook desde el repositorio clonado."
    )
os.chdir(ROOT)
os.environ["COURSE_ROOT"] = str(ROOT)
print(f"Raíz del curso: {ROOT}")

## 1. Contrato de un script

Un script debe declarar:

- **entradas:** tipo, ubicación y restricciones;
- **parámetros:** valores configurables sin editar el cuerpo;
- **salidas:** nombres, formato y condición de validez;
- **dependencias:** programas y versiones;
- **errores:** mensaje, código de salida y política para resultados parciales.

```bash
set -E   # hereda ERR traps
set -e   # detiene ante un comando fallido
set -u   # variable no definida es error
set -o pipefail  # un fallo dentro del pipe no queda oculto
```

In [ ]:
%%bash
set +e
bash -c 'false | true; echo "sin pipefail: $?"'
bash -o pipefail -c 'false | true; echo "con pipefail: $?"'
true

## 2. Leer un script robusto

In [ ]:
from pathlib import Path
script = ROOT / "scripts/module04/fasta_batch_stats.sh"
lines = script.read_text(encoding="utf-8").splitlines()
print("\n".join(f"{i:3d}: {line}" for i, line in enumerate(lines[:90], 1)))

Identifique en el script: `usage`, parser de opciones, `die`, `trap`, comprobación de dependencias, glob seguro, archivo temporal y escritura atómica de la salida.

In [ ]:
import subprocess, shutil

outdir = ROOT / "results/module04"
shutil.rmtree(outdir, ignore_errors=True)
outdir.mkdir(parents=True)
cmd = [
    "bash", "scripts/module04/fasta_batch_stats.sh",
    "-i", "data/module04/inputs",
    "-o", "results/module04/summary.tsv",
    "-l", "results/module04/run.log",
]
result = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)
print("exit:", result.returncode)
print(result.stderr)
print((outdir / "summary.tsv").read_text())

## 3. El error también es una salida

In [ ]:
cmd_bad = [
    "bash", "scripts/module04/fasta_batch_stats.sh",
    "-i", "data/module04/bad_inputs",
    "-o", "results/module04/invalid_summary.tsv",
]
result = subprocess.run(cmd_bad, cwd=ROOT, text=True, capture_output=True)
print("exit:", result.returncode)
print("stderr:\n", result.stderr)
print("¿Existe salida final?", (ROOT / "results/module04/invalid_summary.tsv").exists())

### Checkpoint

El fallo esperado tiene código distinto de cero y un mensaje que identifica el archivo. Una herramienta peligrosa imprime “terminado” o deja una salida parcial con apariencia válida.

## 4. Depurar un script deliberadamente defectuoso

In [ ]:
broken = (ROOT / "scripts/module04/broken_pipeline.sh").read_text(encoding="utf-8")
print(broken)

Encuentre al menos ocho problemas. Pistas: argumentos ausentes, variables sin comillas, glob sin archivos, sobrescritura en cada iteración, ausencia de encabezado, falta de formato y de log, éxito falso, nombre de salida fijo y ninguna validación.

## 5. Patrones reutilizables

```bash
require_file() { [[ -s "$1" ]] || { echo "ERROR: $1" >&2; return 66; }; }
require_cmd()  { command -v "$1" >/dev/null || return 127; }

for file in "$input"/*.fasta; do
  [[ -e "$file" ]] || continue
  process_one "$file"
done
```

Use funciones pequeñas y pruebe cada una. Un log debe complementar, no sustituir, el código de salida.

## Reto

Extienda el procesador con una opción `-f fasta|fastq|auto` o un umbral de longitud. Añada una prueba negativa y documente la semántica de códigos de salida.